# Unconditional SE(3) Diffusion Training

Trains an **unconditional** SE(3) score network on a single protein's MD trajectory.
The model learns the full conformational distribution p(x) and generates new backbone
conformations from pure noise — no source-frame conditioning.

**Workflow:**
1. Configure protein and hyperparameters
2. Load trajectory data; visualise reference MD frames (blue, inline)
3. Train `SE3Module` with TensorBoard weight/gradient monitoring
4. Generate new conformations from noise and compare to a reference frame inline

## Step 1: Environment Setup

In [1]:
# Check environment
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Install dependencies
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio'], check=True)
    subprocess.run(['pip', 'install', '-q',
                    'omegaconf', 'pandas', 'tqdm', 'numpy', 'scipy', 'matplotlib',
                    'lightning', 'mdtraj', 'requests', 'tensorboard',
                    'optuna', 'optuna-integration', 'einops', 'py3Dmol'], check=True)
else:
    # Local: install packages that may not be pre-installed in every environment
    import subprocess
    subprocess.run(
        ['pip', 'install', '-q',
         'tensorboard', 'optuna', 'optuna-integration', 'einops', 'py3Dmol'],
        check=True,
    )

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Running on Google Colab
PyTorch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Ensure the Drive data directory exists (symlink created after repo clone)
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)
    print("Drive mounted")
else:
    # Running locally: just make sure data/ exists
    os.makedirs('data', exist_ok=True)
    print(f"Local mode; data dir: {os.path.abspath('data')}")

Mounted at /content/drive
Drive mounted


## Step 2: Get Code from GitHub

Clone your repository to get the `gen_model` code (no data needed)

In [3]:
import os, subprocess

REPO_URL = "https://github.com/JiwonJJeong/winter-gen-pproject.git"

if IN_COLAB:
    if not os.path.exists('winter-gen-pproject'):
        print(f"Cloning {REPO_URL} ...")
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    # Create data/ symlink inside repo root (after chdir) so all cells find it
    if not os.path.islink('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
        print("data/ -> /content/drive/MyDrive/protein_data/data")
    # Pull latest changes and clear bytecode cache
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    for cmd in [
        'find gen_model -name "*.pyc" -delete',
        'find gen_model -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null; true',
    ]:
        subprocess.run(cmd, shell=True)
    print("Code updated")
else:
    # Already inside the repo; just verify gen_model/ is present
    if not os.path.isdir('gen_model'):
        raise RuntimeError(
            "gen_model/ not found. Run this notebook from the repository root:\n"
            "  cd /path/to/winter-gen-pproject && jupyter notebook"
        )
    print(f"Local repo root: {os.path.abspath('.')}")

import subprocess as _sp
result = _sp.run(['ls', 'gen_model/'], capture_output=True, text=True)
print(result.stdout)
print("Code ready")

Cloning https://github.com/JiwonJJeong/winter-gen-pproject.git ...
data/ -> /content/drive/MyDrive/protein_data/data
Code updated
data
dataset.py
diffusion
hpo.py
inference_conditional.py
inference_unconditional.py
__init__.py
models
README.md
splits
train_base.py
train_conditional.py
train_unconditional.py
utils
video_extrapolation.py

Code ready


## Step 3: Configure Protein and Training

**Customize these settings for your protein:**

In [4]:
from omegaconf import OmegaConf

# ── HPO / LoRA settings ───────────────────────────────────────────────────────
# These are used by Step 8 (HPO) and Step 7 (model creation).
HPO_CONFIG = dict(
    lora_r             = 8,      # LoRA rank: 0 = full fine-tuning, 4/8/16 = LoRA adapters
    lora_alpha         = 0.0,    # 0 → auto-set to 2×lora_r
    n_trials           = 20,     # Optuna trials to run during HPO
    epochs_per_trial   = 5,      # Short epochs per trial (keep small for speed)
    pruner             = 'asha', # 'none' or 'asha' (Asynchronous Successive Halving)
    local_attn_sigma   = 15.0,  # IPA spatial cutoff (Å); ~2% attn beyond this distance
    seq_tfmr_sigma     = 20.0,  # seq-transformer cutoff (residues)
)

# ── Protein / training settings ───────────────────────────────────────────────
protein_config = OmegaConf.create({
    'protein': {
        'name':              '4o66_C',
        'replica':           1,
        'train_early_ratio': 0.3,
        'train_ratio':       0.4,
        'val_ratio':         0.2,
    },
    'training': {
        'batch_size':        8,
        'num_epochs':        100,
        'learning_rate':     1e-4,
        'rot_loss_weight':   1.0,
        'trans_loss_weight': 1.0,
        'psi_loss_weight':   1.0,
    },
    'inference': {
        'num_samples': 5,
        'num_steps':   200,
    },
})
prot_cfg = protein_config.protein

# SE(3) diffusion and model configs are built from default_se3_conf() /
# default_model_conf() in later cells. No coordinate_scaling needed here:
# MDGenDataset computes a dynamic coord_scale from training data statistics.

print("Configuration Summary:")
print("="*70)
print(f"Protein   : {prot_cfg.name}_R{prot_cfg.replica}")
print(f"LoRA      : r={HPO_CONFIG['lora_r']}  (0 = full fine-tuning)")
print(f"HPO       : {HPO_CONFIG['n_trials']} trials × {HPO_CONFIG['epochs_per_trial']} epochs | pruner={HPO_CONFIG['pruner']}")
print(f"Training  : {protein_config.training.num_epochs} epochs | batch={protein_config.training.batch_size} | lr={protein_config.training.learning_rate}")
print("="*70)

# IGSO3 cache: SO3 score lookup table, shared by all HPO trials and full training.
# Storing on Drive makes it a one-time cost instead of ~10 min per Colab session.
IGSO3_CACHE = '/content/drive/MyDrive/mdgen_data/igso3_cache' if IN_COLAB else '/tmp/igso3_cache'


Configuration Summary:
Protein   : 4o66_C_R1
LoRA      : r=8  (0 = full fine-tuning)
HPO       : 20 trials × 5 epochs | pruner=asha
Training  : 100 epochs | batch=8 | lr=0.0001


## Step 4: Create/Load Protein Data

This creates the data structure for your specified protein

In [5]:
import os

# Guard: data/ must be symlinked to Drive before downloading.
# If data/ is not a symlink, it means the Drive mount cell (Step 2) was not run
# first and any download would land in the ephemeral VM filesystem (lost on restart).
if IN_COLAB and not os.path.islink('data'):
    raise RuntimeError(
        "data/ is not linked to Google Drive.\n"
        "Run the 'Mount Google Drive' cell above first, then re-run this cell."
    )

# Use the automatic download and prep script
prot_name = protein_config.protein.name
!python scripts/download_and_prep.py {prot_name} --data_dir ./data --out_dir ./data --suffix _latent

# Setup paths for verification
prot_cfg = protein_config.protein
PROTEIN_FULL_NAME = f"{prot_cfg.name}_R{prot_cfg.replica}"
protein_dir = f'data/{prot_cfg.name}'
traj_path = f'{protein_dir}/{PROTEIN_FULL_NAME}_latent.npy'

if os.path.exists(traj_path):
    print(f"\u2705 Data ready at: {traj_path}")
else:
    print(f"\u274c Error: Data not found at {traj_path}")


Protein Zip: 100% 173M/173M [00:35<00:00, 4.90MiB/s]
Extracting to ./data/4o66_C...
Detected timestep: 10.0 ps
Preprocessing 4o66_C (Replicate 1)...
LEU OXT not found
Saved to ./data/4o66_C/4o66_C_R1_latent.npy
Preprocessing 4o66_C (Replicate 2)...
LEU OXT not found
Saved to ./data/4o66_C/4o66_C_R2_latent.npy
Preprocessing 4o66_C (Replicate 3)...
LEU OXT not found
Saved to ./data/4o66_C/4o66_C_R3_latent.npy
Creating 4-way split (Early: 5.0ns, Ratios: [0.6, 0.2, 0.2])...
Found 3 trajectory file(s) for 4o66_C.
  4o66_C_R1_latent: Total 10001, Timestep 10.0ps
    Early: [0:500], Train: [500:6200], Val: [6200:8100], Test: [8100:10001]
  4o66_C_R2_latent: Total 10001, Timestep 10.0ps
    Early: [0:500], Train: [500:6200], Val: [6200:8100], Test: [8100:10001]
  4o66_C_R3_latent: Total 10001, Timestep 10.0ps
    Early: [0:500], Train: [500:6200], Val: [6200:8100], Test: [8100:10001]
Updated splits saved to gen_model/splits/frame_splits.csv
✅ Data ready at: data/4o66_C/4o66_C_R1_latent.npy


In [6]:
# Reference Trajectory Visualization
# Load a sample of MD frames and show them inline before training.
import numpy as np, pandas as pd, os
from gen_model.utils.structure_utils import (
    save_ca_trajectory_pdb, show_py3dmol_trajectory,
)

# Load residue sequence from atlas CSV
_seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
seqres  = _seq_df.loc[prot_cfg.name, 'seqres']
print(f"Protein : {prot_cfg.name}  ({len(seqres)} residues)")

# Sample 30 evenly-spaced Cα frames from the raw MD trajectory
N_REF  = 30
_arr   = np.lib.format.open_memmap(traj_path, 'r')
_idxs  = np.linspace(0, len(_arr) - 1, N_REF, dtype=int)
ref_ca = _arr[_idxs, :, 1, :].astype(np.float32)          # [N_REF, N, 3]
ref_ca = ref_ca - ref_ca.mean(axis=(0, 1), keepdims=True)  # centre at origin

os.makedirs('outputs', exist_ok=True)
ref_pdb = f'outputs/{prot_cfg.name}_reference.pdb'
save_ca_trajectory_pdb(ref_ca, ref_pdb, seqres)
print(f"Reference PDB saved : {ref_pdb}  ({N_REF} frames)")

try:
    view = show_py3dmol_trajectory(ref_ca, seqres, color='blue', interval=150)
    print(f"\n\u25b6 C\u03b1 trace: {N_REF} reference MD frames (blue), animated:")
    view.show()
except ImportError:
    print("py3Dmol not found \u2014 install with:  pip install py3Dmol")
    print(f"Or open {ref_pdb} in PyMOL / VMD / UCSF ChimeraX")


Protein : 4o66_C  (76 residues)
Reference PDB saved : outputs/4o66_C_reference.pdb  (30 frames)

▶ Cα trace: 30 reference MD frames (blue), animated:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## Step 5: Configure Dataset and Model

In [7]:
import sys
sys.path.insert(0, '.')

from gen_model.train_unconditional    import SE3Module
from gen_model.train_base             import default_se3_conf, default_model_conf
from gen_model.data.dataset           import MDGenDataset
from gen_model.diffusion.se3_diffuser import SE3Diffuser

print("✓ Modules imported")

data_config = OmegaConf.create({
    'data_dir':       'data',
    'atlas_csv':      'gen_model/splits/atlas.csv',
    'train_split':    'gen_model/splits/frame_splits.csv',
    'suffix':         '_latent',
    'frame_interval': None,
    'crop_ratio':     0.95,
    'min_t':          0.01,
    'pep_name':       prot_cfg.name,    # single-protein filter
    'replica':        prot_cfg.replica, # single-replica filter
})

print(f"\nDataset config:")
print(f"  Protein : {data_config.pep_name}_R{data_config.replica}")

✓ Modules imported

Dataset config:
  Protein : 4o66_C_R1


/content/winter-gen-pproject/gen_model/diffusion/so3_diffuser.py:216: SyntaxWarning: invalid escape sequence '\s'
  """Extract \sigma(t) corresponding to chosen sigma schedule.


## Step 6: Load Dataset

In [8]:
# Build SE3 diffuser from canonical defaults (schedule_gamma=1.0, no coordinate_scaling)
print("Initialising SE3Diffuser...")
se3_conf     = default_se3_conf(cache_dir=IGSO3_CACHE)
se3_diffuser = SE3Diffuser(se3_conf)
print("✓ SE3Diffuser ready\n")

# MDGenDataset computes coord_scale dynamically from training data (~1/std of Ca coords)
print("Loading datasets...")
train_dataset = MDGenDataset(
    args=data_config, diffuser=se3_diffuser, mode='train',
    repeat=1, num_consecutive=1, stride=1,
)
val_dataset = MDGenDataset(
    args=data_config, diffuser=se3_diffuser, mode='val',
    repeat=1, num_consecutive=1, stride=1,
)
val_dataset.coord_scale = float(train_dataset.coord_scale)

print(f"\n✓ Training frames  : {len(train_dataset)}")
print(f"✓ Validation frames: {len(val_dataset)}")
print(f"✓ coord_scale      : {train_dataset.coord_scale:.4f}  (computed from data)")

sample     = train_dataset[0]
n_residues = sample['res_mask'].shape[0]
print(f"\nSample keys   : {list(sample.keys())}")
print(f"Residues      : {n_residues}")
print(f"rigids_t shape: {sample['rigids_t'].shape}  (per-residue SE3 frame at time t)")
print(f"t (noise)     : {sample['t']:.4f}")

Initialising SE3Diffuser...
✓ SE3Diffuser ready

Loading datasets...
Computing coordinate scale for 5700 frames...
Dataset train mode: coord_scale = 0.1318
Dataset val mode: coord_scale = 0.1000

✓ Training frames  : 5700
✓ Validation frames: 1900
✓ coord_scale      : 0.1318  (computed from data)

Sample keys   : ['aatype', 'seq_idx', 'chain_idx', 'res_mask', 'fixed_mask', 'rigids_0', 'atom37_pos', 'atom14_pos', 'torsion_angles_sin_cos', 'rigids_t', 'trans_score', 'rot_score', 'trans_score_scaling', 'rot_score_scaling', 't', 'sc_ca_t', 'residx_atom14_to_atom37', 'atom37_mask', 'residue_index', 'name', 'frame_indices', 'centroid', 'coord_scale']
Residues      : 76
rigids_t shape: torch.Size([76, 7])  (per-residue SE3 frame at time t)
t (noise)     : 0.5034


## Step 7: Preview Model Architecture

Builds a preview model with default params to show the architecture. HPO (Step 8) will find the optimal hyperparameters; the full training (Step 9) rebuilds the model with those best params.

In [9]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}\n")

# Build model config with optional LoRA adapters
model_conf = default_model_conf(
    lora_r     = HPO_CONFIG['lora_r'],
    lora_alpha = HPO_CONFIG['lora_alpha'],
)

model_pl = SE3Module(
    model_conf        = model_conf,
    se3_conf          = se3_conf,
    lr                = protein_config.training.learning_rate,
    rot_loss_weight   = protein_config.training.rot_loss_weight,
    trans_loss_weight = protein_config.training.trans_loss_weight,
    psi_loss_weight   = protein_config.training.psi_loss_weight,
)

total     = sum(p.numel() for p in model_pl.parameters())
trainable = sum(p.numel() for p in model_pl.parameters() if p.requires_grad)
print(f"✓ Score Network: {total:,} total params | {trainable:,} trainable ({100*trainable/total:.1f}%)")
print(f"  LoRA r={HPO_CONFIG['lora_r']} → {'LoRA adapters only' if HPO_CONFIG['lora_r'] > 0 else 'full fine-tuning'}")

Device: cuda

[LoRA] r=8, alpha=16.0, targets=['linear_1', 'linear_2', 'linear_kv', 'linear_out', 'linear_q']
[LoRA] trainable params: 114,688 / 9,970,894 (1.15%)
✓ Score Network: 9,970,894 total params | 114,688 trainable (1.2%)
  LoRA r=8 → LoRA adapters only


## Step 8: Hyperparameter Optimisation (HPO)

Runs short Optuna trials (`epochs_per_trial` each) on this specific protein to search for the best combination of learning rate, LoRA rank, loss weights, and noise schedule curvature.
The `best_params` dict is automatically passed to the full training step.

In [ ]:
# =============================================================================
# Step 8: Hyperparameter Optimisation (HPO)
# Runs short Optuna trials — protein-specific, using data_config defined above.
# Searches: lr, lora_r, loss weights.
# Outputs `best_params` dict used by the full training cell in Step 9.
# =============================================================================
import types, os, optuna
from torch.utils.data import DataLoader
from gen_model.hpo import _suggest_hparams, _build_se3_conf, _build_pruner
from gen_model.train_base import default_model_conf
from gen_model.diffusion.se3_diffuser import SE3Diffuser

optuna.logging.set_verbosity(optuna.logging.WARNING)   # suppress per-trial noise

HPO_SAVE_DIR = f'checkpoints/hpo/{prot_cfg.name}'
os.makedirs(HPO_SAVE_DIR, exist_ok=True)

def _hpo_objective(trial):
    import torch, lightning as L
    from lightning.pytorch.callbacks import ModelCheckpoint
    from optuna.integration.pytorch_lightning import PyTorchLightningPruningCallback
    from gen_model.train_unconditional import SE3Module as _Module
    from gen_model.data.dataset import MDGenDataset as _DS

    hp         = _suggest_hparams(trial)
    _se3_conf  = _build_se3_conf(hp, cache_dir=IGSO3_CACHE)
    _mc        = default_model_conf(
        use_temporal_embedding=False,
        lora_r=hp['lora_r'], lora_alpha=hp['lora_alpha'],
        local_attn_sigma=HPO_CONFIG['local_attn_sigma'],
        seq_tfmr_sigma=HPO_CONFIG['seq_tfmr_sigma'],
    )
    _diffuser  = SE3Diffuser(_se3_conf)
    _train = _DS(args=data_config, diffuser=_diffuser, mode="train", repeat=1, num_consecutive=1, stride=1)
    _val   = _DS(args=data_config, diffuser=_diffuser, mode="val",   repeat=1, num_consecutive=1, stride=1)
    _val.coord_scale = float(_train.coord_scale)

    _model = _Module(model_conf=_mc, se3_conf=_se3_conf, lr=hp['lr'],
                     rot_loss_weight=hp['rot_loss_weight'],
                     trans_loss_weight=hp['trans_loss_weight'],
                     psi_loss_weight=hp['psi_loss_weight'])

    _pruning_cb = PyTorchLightningPruningCallback(trial, monitor='val_loss')
    _ckpt_cb    = ModelCheckpoint(
        dirpath=os.path.join(HPO_SAVE_DIR, f'trial_{trial.number}'),
        filename='best', monitor='val_loss', save_top_k=1, mode='min',
    )
    _nw = 2
    _pm = torch.cuda.is_available()
    _trainer = L.Trainer(
        max_epochs=HPO_CONFIG['epochs_per_trial'], accelerator='auto', devices='auto',
        callbacks=[_pruning_cb, _ckpt_cb],
        precision='16-mixed' if _pm else 32,
        enable_progress_bar=False, logger=False,
        limit_val_batches=0.5,   # relative ordering is sufficient for HPO
    )
    try:
        _trainer.fit(_model,
            train_dataloaders=DataLoader(_train, batch_size=protein_config.training.batch_size,
                shuffle=True, num_workers=_nw, pin_memory=_pm, persistent_workers=True),
            val_dataloaders=DataLoader(_val, batch_size=protein_config.training.batch_size,
                shuffle=False, num_workers=_nw, pin_memory=_pm, persistent_workers=True))
    except optuna.exceptions.TrialPruned:
        raise
    return _trainer.callback_metrics['val_loss'].item()

_pruner = _build_pruner(types.SimpleNamespace(
    pruner=HPO_CONFIG['pruner'], asha_min_resource=1, asha_reduction_factor=3))
study = optuna.create_study(
    direction='minimize',
    storage=f'sqlite:///{HPO_SAVE_DIR}/optuna.db',
    study_name=f'se3_{prot_cfg.name}',
    load_if_exists=True, pruner=_pruner,
)
def _trial_callback(study, trial):
    status = 'pruned' if trial.value is None else f'val_loss={trial.value:.6f}'
    best   = f'{study.best_value:.6f}' if study.best_trial else 'n/a'
    print(f"  Trial {trial.number + 1:2d}/{HPO_CONFIG['n_trials']}  "
          f"{status}  |  best so far: {best}")

print(f"Running {HPO_CONFIG['n_trials']} HPO trials ({HPO_CONFIG['epochs_per_trial']} epochs each)  protein={prot_cfg.name}...")
study.optimize(_hpo_objective, n_trials=HPO_CONFIG['n_trials'],
               catch=(Exception,), callbacks=[_trial_callback])

# Extract best params
_bt = study.best_trial
best_params = dict(_bt.params)
best_params['lora_alpha'] = float(2 * best_params.get('lora_r', 0))

print(f"\n{'='*62}")
print(f"HPO complete  {len(study.trials)} trials | best val_loss = {_bt.value:.6f}  (trial #{_bt.number})")
print(f"{'='*62}")
print("Best params (passed automatically to full training in Step 9):")
for _k, _v in best_params.items():
    print(f"  {_k:25s}: {_v}")

Running 20 HPO trials (5 epochs each)  protein=4o66_C...
Computing coordinate scale for 5700 frames...
Dataset train mode: coord_scale = 0.1318
Dataset val mode: coord_scale = 0.1000


INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[LoRA] r=8, alpha=16.0, targets=['linear_1', 'linear_2', 'linear_kv', 'linear_out', 'linear_q']
[LoRA] trainable params: 114,688 / 9,970,894 (1.15%)


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ScoreNetwork │ 10.0 M │ train │     0 │
└───┴───────┴──────────────┴────────┴───────┴───────┘

...

  Trial 16/20  val_loss=1.806404  |  best so far: 1.728166
Computing coordinate scale for 5700 frames...
Dataset train mode: coord_scale = 0.1316
Dataset val mode: coord_scale = 0.1000


INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_R

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ScoreNetwork │  9.9 M │ train │     0 │
└───┴───────┴──────────────┴────────┴───────┴───────┘

Trainable params: 9.9 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 9.9 M                                                                                                
Total estimated model params size (MB): 39                                                                         
Modules in train mode: 227                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


## Step 9: Full Training (HPO Best Params)

In [ ]:
import lightning as L

class WeightHistogramCallback(L.Callback):
    """Log per-parameter weight and gradient histograms to TensorBoard each epoch.

    After training, open TensorBoard and navigate to:
      Histograms   — weight distributions for every named layer, over epochs
      Distributions — alternative time-series view of the same data

    Gradients are logged only when they exist (i.e. after at least one backward pass).
    Large models may generate many histogram series — filter by layer name in TensorBoard.
    """

    def on_train_epoch_end(self, trainer, pl_module):
        # Support both single logger and list-of-loggers
        loggers = trainer.loggers if hasattr(trainer, 'loggers') else [trainer.logger]
        tb = None
        for lg in loggers:
            if hasattr(lg, 'experiment') and hasattr(lg.experiment, 'add_histogram'):
                tb = lg.experiment
                break
        if tb is None:
            return

        epoch = trainer.current_epoch
        for name, param in pl_module.named_parameters():
            if not param.requires_grad:
                continue
            tb.add_histogram(f'weights/{name}', param.detach().cpu(), epoch)
            if param.grad is not None:
                tb.add_histogram(f'grads/{name}', param.grad.detach().cpu(), epoch)

print("\u2713 WeightHistogramCallback defined")

In [ ]:
# ── TensorBoard inline viewer ─────────────────────────────────────────────────
# Tabs to look at:
#   Scalars      → train_loss_step / val_loss curves over training
#   Histograms   → per-parameter weight and gradient distributions, per epoch
#   Distributions → same data as Histograms in a violin/area view
#
# Can be run before training finishes — TensorBoard reads logs incrementally.
# For a standalone browser tab instead:  tensorboard --logdir tb_logs/
# ─────────────────────────────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir tb_logs/

In [ ]:
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers   import TensorBoardLogger, CSVLogger
from torch.utils.data import DataLoader
from gen_model.hpo        import _build_se3_conf
from gen_model.train_base import default_model_conf, default_se3_conf
from gen_model.diffusion.se3_diffuser import SE3Diffuser
from gen_model.train_unconditional import SE3Module
from gen_model.data.dataset import MDGenDataset

# ── Hyperparameters: use HPO best if available, else fall back to defaults ────
if 'best_params' in vars() and best_params:
    print("\u2713 Using HPO best hyperparameters for full training:")
    _lr  = best_params['lr']
    _rot_w, _trans_w, _psi_w  = best_params['rot_loss_weight'], best_params['trans_loss_weight'], best_params['psi_loss_weight']
    _se3_conf = _build_se3_conf(best_params, cache_dir=IGSO3_CACHE)
else:
    print("HPO not run \u2014 using defaults from configuration cell:")
    _lr  = protein_config.training.learning_rate
    _rot_w, _trans_w, _psi_w  = protein_config.training.rot_loss_weight, protein_config.training.trans_loss_weight, protein_config.training.psi_loss_weight
    _se3_conf = default_se3_conf(cache_dir=IGSO3_CACHE)

# Full training always uses the complete model — no LoRA.
# HPO ran with LoRA for speed; the optimal lr / loss-weights / schedule transfer directly.
_lora_r, _lora_alpha = 0, 0.0
_local_attn_sigma = 15.0  # Å cutoff; ~2% attention at this distance. 0 = global. Typical: 12–20 Å.
_seq_tfmr_sigma   = 20.0  # residue cutoff; ~2% attention at this separation. 0 = global. Typical: 16–32.

print(f"  lr={_lr:.2e}  rot_w={_rot_w:.2f}  trans_w={_trans_w:.2f}  psi_w={_psi_w:.2f}  (full model, lora_r=0)")

# ── Rebuild datasets with HPO-chosen SE3 schedule ────────────────────────────
print("\nRebuilding datasets with chosen se3_conf...")
_diffuser = SE3Diffuser(_se3_conf)
_train_ds = MDGenDataset(args=data_config, diffuser=_diffuser, mode="train",
                         repeat=1, num_consecutive=1, stride=1)
_val_ds   = MDGenDataset(args=data_config, diffuser=_diffuser, mode="val",
                         repeat=1, num_consecutive=1, stride=1)
_val_ds.coord_scale = float(_train_ds.coord_scale)
train_loader = DataLoader(_train_ds, batch_size=protein_config.training.batch_size, shuffle=True)
val_loader   = DataLoader(_val_ds,   batch_size=protein_config.training.batch_size, shuffle=False)
print(f"\u2713 Train: {len(_train_ds)} frames | Val: {len(_val_ds)} frames")

# ── Build model with HPO best params ─────────────────────────────────────────
_mc = default_model_conf(use_temporal_embedding=False, lora_r=_lora_r, lora_alpha=_lora_alpha, local_attn_sigma=_local_attn_sigma, seq_tfmr_sigma=_seq_tfmr_sigma)
model_pl = SE3Module(
    model_conf=_mc, se3_conf=_se3_conf, lr=_lr,
    rot_loss_weight=_rot_w, trans_loss_weight=_trans_w, psi_loss_weight=_psi_w,
)
_tot = sum(p.numel() for p in model_pl.parameters())
_trn = sum(p.numel() for p in model_pl.parameters() if p.requires_grad)
print(f"\u2713 Model: {_tot:,} params | {_trn:,} trainable ({100*_trn/_tot:.1f}%)")
print(f"  LoRA r={_lora_r} \u2192 {'adapters only' if _lora_r > 0 else 'full fine-tuning'}  |  local_attn_sigma={_local_attn_sigma} \u00c5  seq_tfmr_sigma={_seq_tfmr_sigma}")

# ── Loggers ───────────────────────────────────────────────────────────────────
_tb_logger  = TensorBoardLogger('tb_logs', name=prot_cfg.name)
_csv_logger = CSVLogger('lightning_logs')
print(f"\nTensorBoard logs : tb_logs/{prot_cfg.name}/version_*/")

# ── Checkpointing ─────────────────────────────────────────────────────────────
_ckpt_cb = ModelCheckpoint(
    dirpath=f'checkpoints/{prot_cfg.name}_se3',
    filename='se3-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3, monitor='val_loss', mode='min', save_last=True,
)

# ── Train (full run with HPO best params) ─────────────────────────────────────
trainer = L.Trainer(
    max_epochs=protein_config.training.num_epochs,
    accelerator="auto", devices=1,
    logger=[_csv_logger, _tb_logger],
    callbacks=[_ckpt_cb, WeightHistogramCallback()],
    precision="16-mixed" if __import__('torch').cuda.is_available() else 32,
)
trainer.fit(model_pl, train_dataloaders=train_loader, val_dataloaders=val_loader)

import glob

ckpt_dir  = f'checkpoints/{prot_cfg.name}_se3'
ckpt_list = sorted(glob.glob(f'{ckpt_dir}/*.ckpt'))

if ckpt_list:
    print(f"Checkpoints in {ckpt_dir}/:")
    for p in ckpt_list:
        print(f"  {p}")
    # Prefer non-last checkpoints (ranked by val_loss in filename)
    ranked = [p for p in ckpt_list if 'last' not in p]
    best_ckpt = ranked[-1] if ranked else ckpt_list[-1]
    print(f"\nBest checkpoint : {best_ckpt}")
else:
    best_ckpt = None
    print(f"No checkpoints found in {ckpt_dir}/")
    print("Run Step 8 (training) first.")

## Step 10: Evaluate

In [ ]:
# =============================================================================
# Inline Visualization: Condition Frame vs. Generated Frame
#
# Shows ONE reference MD frame (blue) overlaid with ONE conformation generated
# by the unconditional model from pure noise (orange).
# seqres is recovered from the aatype tensor via aatype_to_seqres — no CSV.
# =============================================================================
import glob, torch, os, numpy as np
from gen_model.train_base              import default_se3_conf, default_model_conf
from gen_model.train_unconditional     import SE3Module
from gen_model.diffusion.se3_diffuser  import SE3Diffuser
from gen_model.inference_unconditional import run_reverse_sde
from gen_model.inference_conditional   import compute_coord_scale
from gen_model.data.residue_constants  import restype_order
from gen_model.utils.structure_utils   import (
    aatype_to_seqres, rigids_to_ca_trajectory,
    save_ca_trajectory_pdb, show_py3dmol_overlay,
)

SOURCE_FRAME_IDX = 0    # MD frame index to use as the reference frame
NUM_STEPS        = 100  # reverse-SDE steps (increase for better quality)

_device   = 'cuda' if torch.cuda.is_available() else 'cpu'
_ckpt_dir = f'checkpoints/{prot_cfg.name}_se3'
_ckpts    = [p for p in sorted(glob.glob(f'{_ckpt_dir}/*.ckpt')) if 'last' not in p]
_ckpt     = _ckpts[-1] if _ckpts else None

if _ckpt is None:
    print("No checkpoint found \u2014 train the model first (Step 9).")
else:
    print(f"Checkpoint   : {_ckpt}")

    # \u2500\u2500 Condition frame (one real MD frame) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    _arr    = np.lib.format.open_memmap(traj_path, 'r')
    src_ca  = _arr[SOURCE_FRAME_IDX, :, 1, :].astype(np.float32)  # [N, 3]
    src_ca -= src_ca.mean(axis=0, keepdims=True)
    src_ca  = src_ca[None]                                          # [1, N, 3]

    # \u2500\u2500 Build aatype (used for inference; seqres recovered from it) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    # seqres may already be in scope from the reference viz cell; if not, use aatype.
    if 'seqres' not in vars():
        import pandas as pd
        seqres = pd.read_csv('gen_model/splits/atlas.csv', index_col='name').loc[prot_cfg.name, 'seqres']
    _aatype   = torch.tensor([restype_order[c] for c in seqres], dtype=torch.long)
    _res_mask = torch.ones(len(seqres), dtype=torch.float32)

    # \u2500\u2500 Generated frame (from pure noise) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
    _se3_conf   = default_se3_conf(cache_dir=IGSO3_CACHE)
    _model_conf = default_model_conf(lora_r=HPO_CONFIG['lora_r'], lora_alpha=HPO_CONFIG['lora_alpha'])
    _module = SE3Module.load_from_checkpoint(
        _ckpt, model_conf=_model_conf, se3_conf=_se3_conf, map_location=_device,
    ).to(_device).eval()
    _diffuser    = SE3Diffuser(_se3_conf)
    _coord_scale = compute_coord_scale(traj_path)

    print(f"coord_scale  : {_coord_scale:.4f}")
    print(f"Generating one sample ({NUM_STEPS} steps) on {_device}\u2026")

    _rigid  = run_reverse_sde(
        model=_module, diffuser=_diffuser, aatype=_aatype, res_mask=_res_mask,
        N=len(seqres), num_steps=NUM_STEPS, device=_device,
    )
    gen_ca  = rigids_to_ca_trajectory([_rigid], _coord_scale)  # [1, N, 3]

    # seqres is recovered from the inference aatype — no extra CSV lookup
    _seqres_for_viz = aatype_to_seqres(_aatype)

    os.makedirs('outputs', exist_ok=True)
    src_pdb = f'outputs/{prot_cfg.name}_source_frame{SOURCE_FRAME_IDX}.pdb'
    gen_pdb = f'outputs/{prot_cfg.name}_generated_frame.pdb'
    save_ca_trajectory_pdb(src_ca, src_pdb, _seqres_for_viz)
    save_ca_trajectory_pdb(gen_ca, gen_pdb, _seqres_for_viz)
    print(f"Source PDB   : {src_pdb}  |  Generated : {gen_pdb}")

    try:
        view = show_py3dmol_overlay(
            [src_ca[0], gen_ca[0]], _seqres_for_viz,
            colors=['blue', 'orange'],
        )
        print("\n\u25b6 Overlay \u2014 blue: real MD frame | orange: unconditional model output")
        view.show()
    except ImportError:
        print("\npy3Dmol not installed \u2014 install with:  pip install py3Dmol")
        print(f"Source PDB : {src_pdb}  |  Generated : {gen_pdb}")


In [ ]:
# Inference runs the reverse SE(3) SDE from pure noise to generate new conformations.
# Run from terminal after training completes:

npy_path = f'data/{prot_cfg.name}/{prot_cfg.name}_R{prot_cfg.replica}_latent.npy'

if best_ckpt:
    print("To generate samples, run from terminal:")
    print()
    print(f"  python gen_model/inference_unconditional.py \\")
    print(f"      --checkpoint {best_ckpt} \\")
    print(f"      --npy_path {npy_path} \\")
    print(f"      --protein_name {prot_cfg.name} \\")
    print(f"      --num_steps {protein_config.inference.num_steps} \\")
    print(f"      --num_samples {protein_config.inference.num_samples} \\")
    print(f"      --out_dir outputs/{prot_cfg.name}_se3")
else:
    print("Train a model first (Step 8), then run inference.")


## Step 10: Generate Samples

In [ ]:
import matplotlib.pyplot as plt, pandas as pd, glob

# Lightning's CSVLogger writes metrics to lightning_logs/version_N/metrics.csv
log_files = sorted(glob.glob('lightning_logs/version_*/metrics.csv'))

if log_files:
    df  = pd.read_csv(log_files[-1])
    val = df[['epoch', 'val_loss']].dropna()
    trn = df[['step',  'train_loss_step']].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if not val.empty:
        axes[0].plot(val['epoch'], val['val_loss'], 'o-', color='steelblue')
        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Loss')
        axes[0].set_title(f'Validation Loss — {prot_cfg.name}'); axes[0].grid(alpha=0.3)
    if not trn.empty:
        axes[1].plot(trn['step'], trn['train_loss_step'], alpha=0.6, color='steelblue')
        axes[1].set_xlabel('Step'); axes[1].set_ylabel('Train Loss (step)')
        axes[1].set_title('Training Loss'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f"Log: {log_files[-1]}")
else:
    print("No training logs found. Run Step 8 (training) first.")
    print("Expected location: lightning_logs/version_*/metrics.csv")

## Step 11: Visualize

In [ ]:
import os, subprocess

save_zip = f'{prot_cfg.name}_results.zip'
dirs_to_pack = [d for d in [ckpt_dir, f'outputs/{prot_cfg.name}_se3'] if os.path.isdir(d)]

if dirs_to_pack:
    subprocess.run(['zip', '-rq', save_zip] + dirs_to_pack, check=True)
    print(f"✓ Packaged: {save_zip}  ({', '.join(dirs_to_pack)})")
    if IN_COLAB:
        from google.colab import files
        files.download(save_zip)
        print("✓ Download started")
    else:
        print(f"✓ Saved locally as {save_zip}")
else:
    print("Nothing to package yet — run Steps 8 and 10 first.")

## Step 12: Download Results

In [ ]:
# Package results
!zip -rq {prot_cfg.name}_results.zip {save_dir} {output_dir}

print(f"\nResults packaged: {prot_cfg.name}_results.zip")
print(f"  Checkpoints: {save_dir}/")
print(f"  Outputs: {output_dir}/")

if IN_COLAB:
    from google.colab import files
    files.download(f'{prot_cfg.name}_results.zip')
    print("\n✓ Download started")
else:
    print(f"\n✓ Saved locally as {prot_cfg.name}_results.zip")

## Summary

**What we did:**
1. ✓ Configured protein: `{protein_config.protein.name}_R{protein_config.protein.replica}`
2. ✓ Downloaded and preprocessed trajectory data using
download_and_prep.py
3. ✓ Trained DDPM for {protein_config.training.num_epochs} epochs
4. ✓ Evaluated denoising on validation frames
5. ✓ Generated new conformations from noise

**Key insight:** The model learned the conformational space of **one specific protein** by training on different frames from its MD trajectory.

**To train on a different protein:**
- Change `protein_config.protein.name`
- Rerun from Step 3 onwards